In [3]:
from sage.all import *
from sage.misc.prandom import sample, shuffle, random

load("alpha_od.sage")

############################################################
# GLOBAL CACHE
############################################################

_alpha_cache = {}

def alpha_cached(G):
    key = G.canonical_label().graph6_string()
    if key in _alpha_cache:
        return _alpha_cache[key]
    a = alpha_od_ilp_correct(G)
    _alpha_cache[key] = a
    return a

############################################################
# STRONG ODD INDEPENDENT SET
############################################################

def is_strong_odd_independent_set(G, S):
    S = set(S)

    # independent
    for u in S:
        for v in G.neighbors(u):
            if v in S:
                return False

    # odd domination condition
    for u in G.vertices():
        if u not in S:
            c = sum(1 for v in G.neighbors(u) if v in S)
            if c % 2 == 0:
                return False

    return True

############################################################
# QUICK GRAPH FILTERS
############################################################

def basic_regular_connected(G):
    if not G.is_connected():
        return False

    deg = G.degree()
    if len(set(deg)) != 1:
        return False

    r = deg[0]
    if r % 2 == 0:
        return False

    return True

def triangle_free(G):
    return G.clique_number() < 3

def diameter_ok(G):
    try:
        return G.diameter() <= 3
    except:
        return False

############################################################
# CONDITIONS
############################################################

# (i) every neighborhood is maximum strong odd independent
def satisfies_i(G):

    if not basic_regular_connected(G): return False
    if not triangle_free(G): return False
    if not diameter_ok(G): return False

    r = G.degree()[0]
    alpha = alpha_cached(G)

    for v in G.vertices():
        Nv = list(G.neighbors(v))
        if len(Nv) != r:
            return False
        if not is_strong_odd_independent_set(G, Nv):
            return False
        if len(Nv) != alpha:
            return False

    return True


# (ii) some neighborhood is maximum strong odd independent
def satisfies_ii(G):

    if not basic_regular_connected(G): return False
    if not triangle_free(G): return False
    if not diameter_ok(G): return False

    alpha = alpha_cached(G)

    for v in G.vertices():
        Nv = list(G.neighbors(v))
        if is_strong_odd_independent_set(G, Nv) and len(Nv) == alpha:
            return True

    return False


# (iii) alpha_od(G) = r
def satisfies_iii(G):

    if not basic_regular_connected(G): return False

    r = G.degree()[0]
    n = G.num_verts()

    # theoretical bound
    if r >= 3 and n > r*(r*r-1):
        return False

    try:
        return alpha_cached(G) == r
    except:
        return False

############################################################
# HEURISTIC SCORE (guides random walk)
############################################################

def heuristic_score(G):

    score = 0

    if basic_regular_connected(G): score += 3
    if triangle_free(G): score += 3

    try:
        d = G.diameter()
        score += max(0, 4-d)
    except:
        pass

    # reward strong odd neighborhoods
    for v in G.vertices():
        if is_strong_odd_independent_set(G, G.neighbors(v)):
            score += 2

    # lightweight alpha reward
    if score >= 5:
        try:
            score += min(6, alpha_cached(G))
        except:
            pass

    return score

############################################################
# DEGREE PRESERVING SWITCH
############################################################

def local_triangle_check(G, verts):
    for v in verts:
        Nv = set(G.neighbors(v))
        for u in Nv:
            if Nv.intersection(G.neighbors(u)):
                return True
    return False

def random_2switch(G, trials=40):

    edges = list(G.edges(labels=False))
    if len(edges) < 2:
        return None

    for _ in range(trials):

        (a,b),(c,d) = sample(edges,2)
        if len({a,b,c,d}) < 4:
            continue

        swaps = [((a,c),(b,d)),((a,d),(b,c))]
        shuffle(swaps)

        for (x,y),(u,v) in swaps:

            if G.has_edge(x,y) or G.has_edge(u,v):
                continue

            H = G.copy()
            H.delete_edge(a,b)
            H.delete_edge(c,d)
            H.add_edge(x,y)
            H.add_edge(u,v)

            if not H.is_connected():
                continue

            if local_triangle_check(H,[a,b,c,d,x,y,u,v]):
                continue

            return H

    return None

############################################################
# SIMULATED ANNEALING
############################################################

def annealing_walk(G0, steps=6000, T0=5.0, cool=0.999):

    G = G0.copy()
    best = G.copy()

    current = heuristic_score(G)
    best_score = current
    T = T0

    for step in range(steps):

        H = random_2switch(G)
        if H is None:
            continue

        new = heuristic_score(H)
        delta = new-current

        if delta >= 0 or random() < exp(delta/max(T,0.001)):
            G = H
            current = new

        if new > best_score:
            best = H.copy()
            best_score = new

        T *= cool

        if step % 1500 == 0 and step>0:
            T = T0

    return best

############################################################
# MAIN SEARCH
############################################################

def task2_research_search(G0, restarts=20, steps=6000):

    if not G0.is_regular():
        raise ValueError("Initial graph must be r-regular")

    found = {"i":[], "ii":[], "iii":[]}
    seen = set()

    for r in range(restarts):

        print(f"\n=== restart {r} ===")

        G = annealing_walk(G0, steps)

        key = G.canonical_label().graph6_string()
        if key in seen:
            continue
        seen.add(key)

        ok_i   = satisfies_i(G)
        ok_ii  = satisfies_ii(G)
        ok_iii = satisfies_iii(G)

        if ok_i or ok_ii or ok_iii:
            print("FOUND interesting graph:",
                  "n=",G.num_verts(),
                  "r=",G.degree()[0],
                  "alpha=",alpha_cached(G))

        if ok_i:
            print("  satisfies (i)")
            found["i"].append((G.copy(),key))

        if ok_ii:
            print("  satisfies (ii)")
            found["ii"].append((G.copy(),key))

        if ok_iii:
            print("  satisfies (iii)")
            found["iii"].append((G.copy(),key))

    print("\nSUMMARY:")
    for k in found:
        print(k,len(found[k]))

    return found

############################################################
# SAVE RESULTS
############################################################

def save_graph_collections(found_dict, prefix="task2", per_pdf=8):

    for condition, graphs in found_dict.items():

        if len(graphs)==0:
            print("No graphs for",condition)
            continue

        for start in range(0,len(graphs),per_pdf):

            chunk = graphs[start:start+per_pdf]
            plots=[]

            for i,(G,key) in enumerate(chunk,start=start+1):

                p = G.plot(vertex_size=30,
                           edge_thickness=2,
                           vertex_labels=False,
                           layout="spring",
                           figsize=4)

                plots.append(p)

                with open(f"{prefix}_{condition}_{i}.graph6","w") as f:
                    f.write(key)

            graphics_array(plots,ncols=2).save(
                f"{prefix}_{condition}_{start//per_pdf+1}.pdf"
            )

In [4]:
G0 = graphs.CompleteBipartiteGraph(5,5)  # odd r example family
found = task2_research_search(G0, restarts=40, steps=8000)
save_graph_collections(found)


=== restart 0 ===


FOUND interesting graph: n= 10 r= 5 alpha= 5
  satisfies (i)
  satisfies (ii)
  satisfies (iii)

=== restart 1 ===



=== restart 2 ===



=== restart 3 ===



=== restart 4 ===


...


=== restart 36 ===



=== restart 37 ===



=== restart 38 ===



=== restart 39 ===



SUMMARY:
i 1
ii 1
iii 1
